# Clase 140 — Análisis de sentimiento

Clasificación de texto (la tarea NLP clásica, IMDB) con el pipeline
`TextVectorization → Embedding → cuerpo → Dense(1, sigmoid)`. Comparamos varias
arquitecturas: **bag-of-embeddings**, **Conv1D** y **Bidirectional LSTM**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Corpus (en producción: `keras.datasets.imdb`)

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
# Mini-corpus de juguete; el dataset real es IMDB (25k train / 25k test).
textos = np.array([
    "la pelicula fue excelente y muy entretenida",
    "una obra maestra, actuaciones brillantes",
    "pesima, aburrida y sin sentido alguno",
    "no la recomiendo, perdi mi tiempo",
    "genial, la volveria a ver sin dudarlo",
    "horrible guion y mala direccion",
])
etiquetas = np.array([1, 1, 0, 0, 1, 0], dtype="float32")   # 1 = positivo, 0 = negativo
print("ejemplos:", len(textos))

## 2. `TextVectorization`: tokenizar y padear

In [ ]:
MAX_TOKENS, SEQ_LEN = 20_000, 200
vectorizador = layers.TextVectorization(max_tokens=MAX_TOKENS,
                                        output_sequence_length=SEQ_LEN)
vectorizador.adapt(textos)                  # aprende el vocabulario desde el corpus
print("vocab aprendido:", len(vectorizador.get_vocabulary()))
print("review vectorizada (primeros 10):", vectorizador(textos[:1]).numpy()[0, :10])

## 3. Bag-of-embeddings (mean pooling, ignora el orden)

In [ ]:
def construir(cuerpo, mask=True):
    entrada = keras.Input(shape=(1,), dtype="string")
    x = vectorizador(entrada)
    x = layers.Embedding(MAX_TOKENS, 64, mask_zero=mask)(x)   # lookup table entrenable
    x = cuerpo(x)
    salida = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(entrada, salida)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

bag = construir(lambda x: layers.GlobalAveragePooling1D()(x))   # promedia embeddings
bag.summary()

## 4. Conv1D para texto (capta n-gramas)

In [ ]:
cnn = construir(
    lambda x: layers.GlobalMaxPooling1D()(layers.Conv1D(64, 5, activation="relu")(x)),
    mask=False,   # Conv1D no consume máscara -> se desactiva mask_zero
)
cnn.summary()

## 5. Bidirectional LSTM (contexto bidireccional)

In [ ]:
bilstm = construir(lambda x: layers.Bidirectional(layers.LSTM(64))(x), mask=True)
bilstm.summary()
# mask_zero=True + LSTM: el padding (0) se ignora automáticamente.

## 6. Entrenar el pipeline end-to-end (texto crudo como input)

In [ ]:
bag.fit(textos, etiquetas, epochs=3, verbose=2)
acc = float(bag.evaluate(textos, etiquetas, verbose=0)[1])
print("accuracy final:", round(acc, 3))
# El modelo recibe strings directamente: TextVectorization vive dentro del grafo.

## Ejercicios

1. **Baseline clásico**: `TfidfVectorizer + LogisticRegression` como referencia (~0.88).
2. **Bag vs Conv1D vs BiLSTM**: entrená las tres sobre IMDB y compará accuracy.
3. **`mask_zero`**: mostrá el efecto de activar/desactivar el masking en la BiLSTM.
4. **Pre-trained**: inicializá el `Embedding` con GloVe 100d y compará contra random.

## Conclusiones

- `TextVectorization` tokeniza y padea; puede vivir **dentro** del modelo (input = strings).
- El `Embedding` es una **lookup table entrenable** `(vocab, dim)`.
- **Bag-of-embeddings** ignora el orden; **Conv1D** capta n-gramas; **BiLSTM** capta contexto largo.
- `mask_zero=True` hace que LSTM/GRU ignoren el padding; **Conv1D no soporta máscara**.
- En 2026 lo práctico es Hugging Face (DistilBERT, ~94%); esta clase es pedagógica.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — Baseline clásico TF-IDF + LogisticRegression (sklearn, verificable)

La referencia a superar (~0.88 en IMDB). Aquí se corre sobre un mini-corpus como demo autónoma con `assert`.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

textos = np.array([
    "excelente pelicula muy entretenida y genial",
    "obra maestra actuaciones brillantes",
    "buenisima la volveria a ver, genial",
    "pesima aburrida y sin sentido",
    "horrible guion mala direccion",
    "no la recomiendo, perdi el tiempo"])
etiquetas = np.array([1, 1, 1, 0, 0, 0])

pipe = TfidfVectorizer().fit(textos)
clf = LogisticRegression().fit(pipe.transform(textos), etiquetas)
acc = clf.score(pipe.transform(textos), etiquetas)
print("accuracy (mini-corpus):", acc, "| en IMDB real ~0.88")
assert acc >= 0.8

### Ejercicio 2 — Bag vs Conv1D vs BiLSTM

Mismo `Embedding`, distinto cuerpo: bag-of-embeddings (ignora orden), Conv1D (n-gramas) y BiLSTM (contexto bidireccional).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

MAX_TOKENS, SEQ_LEN = 20_000, 200
vectorizador = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
# vectorizador.adapt(train_textos)

def construir(cuerpo, mask=True):
    e = keras.Input(shape=(1,), dtype="string")
    x = vectorizador(e)
    x = layers.Embedding(MAX_TOKENS, 64, mask_zero=mask)(x)
    x = cuerpo(x)
    m = keras.Model(e, layers.Dense(1, activation="sigmoid")(x))
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]); return m

bag = construir(lambda x: layers.GlobalAveragePooling1D()(x))
cnn = construir(lambda x: layers.GlobalMaxPooling1D()(layers.Conv1D(64, 5, activation="relu")(x)), mask=False)
bilstm = construir(lambda x: layers.Bidirectional(layers.LSTM(64))(x))
print("3 arquitecturas listas: bag / Conv1D / BiLSTM")

### Ejercicio 3 — Efecto de `mask_zero` en la BiLSTM

`mask_zero=True` hace que la LSTM **ignore el padding** (token 0); sin él, procesa cientos de ceros y degrada. (Conv1D no consume máscara.)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

MAX_TOKENS = 20_000
def bilstm(mask):
    e = keras.Input(shape=(1,), dtype="string")
    x = vectorizador(e)
    x = layers.Embedding(MAX_TOKENS, 64, mask_zero=mask)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    return keras.Model(e, layers.Dense(1, activation="sigmoid")(x))

con_mask, sin_mask = bilstm(True), bilstm(False)
print("mask_zero=True ignora el padding -> mejor señal en secuencias cortas padeadas")

### Ejercicio 4 — Embedding pre-entrenado con GloVe 100d

Se inicializa la matriz del `Embedding` con vectores GloVe y se compara contra random. Con datasets chicos, GloVe arranca con mejor accuracy.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# 1) cargar GloVe (glove.6B.100d.txt): palabra -> vector 100d
# emb_index = {}
# for linea in open("glove.6B.100d.txt", encoding="utf-8"):
#     partes = linea.split(); emb_index[partes[0]] = np.asarray(partes[1:], "float32")

# 2) armar la matriz alineada al vocab del vectorizador
vocab = vectorizador.get_vocabulary()
matriz = np.zeros((len(vocab), 100), dtype="float32")
# for i, palabra in enumerate(vocab):
#     if palabra in emb_index: matriz[i] = emb_index[palabra]

emb = layers.Embedding(len(vocab), 100,
                       embeddings_initializer=keras.initializers.Constant(matriz),
                       trainable=True)          # trainable=False = feature fijo
print("Embedding inicializado con GloVe 100d (vs random) -> mejor arranque")